# 04 — Multi-Agent Systems: Composition Patterns

## What You'll Learn
- The four multi-agent composition patterns
- SequentialAgent: linear pipelines (A→B→C)
- ParallelAgent: concurrent execution (A+B+C at once)
- LoopAgent: iterative refinement (repeat until condition)
- Sub-agent routing: LLM decides which specialist handles the request
- Nested composition: combining patterns

> **Prerequisites:** Notebooks 02 (Agent deep dive), 03 (Tools)


## 4.1 Why Multiple Agents?

A single agent with many tools works for simple tasks. But as complexity
grows, you hit limits:

| Problem | Multi-Agent Solution |
|---|---|
| Instruction gets too long | Each agent has focused instructions |
| Context window fills up | Each agent has its own context |
| Conflicting tool behaviors | Each agent has only relevant tools |
| Hard to test individual behaviors | Test each agent independently |
| Need parallel execution | ParallelAgent runs multiple at once |

### The Four Composition Patterns

```
1. SEQUENTIAL   A → B → C         (pipeline)
2. PARALLEL     A + B + C         (concurrent, same input)
3. LOOP         repeat A until X  (iterative refinement)
4. ROUTING      LLM picks A, B, or C  (dynamic delegation)
```


In [2]:
import asyncio
from google.adk import Agent
from google.adk.runners import Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai.types import Content, Part
print("✅ Core imports ready")
import os
from dotenv import load_dotenv, find_dotenv

# Load the .env file
load_dotenv(find_dotenv())
print(f'API key: {"✅" if os.environ.get("GOOGLE_API_KEY") else "❌"}')
from google.adk.agents.sequential_agent import SequentialAgent
from google.adk.agents.parallel_agent import ParallelAgent
from google.adk.agents.loop_agent import LoopAgent
print("✅ Multi-agent imports ready")


✅ Core imports ready
API key: ❌
✅ Multi-agent imports ready


## 4.2 SequentialAgent — Pipeline Processing

Runs sub-agents **one after another**. Output of Agent 1 is available
to Agent 2 via `session.state[output_key]`.

**When to use:** Any linear workflow — extract→transform→load,
research→fact-check→summarize, brainstorm→outline→draft.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SEQUENTIAL PIPELINE — Extract → Categorize → Summarize
# ═══════════════════════════════════════════════════════════════════

# ─── Stage 1: Extract facts ──────────────────────────────────────
extractor = Agent(
    name="extractor",
    model="gemini-2.5-flash",
    instruction="Extract 3-5 key facts from the input. Bullet points only. Be concise.",
    output_key="facts",  # ← Saved to session.state["facts"] for next agent
)

# ─── Stage 2: Categorize facts ───────────────────────────────────
# This agent reads session.state["facts"] from the previous stage
categorizer = Agent(
    name="categorizer",
    model="gemini-2.5-flash",
    instruction="Read the extracted facts. Categorize each as: Technology, Business, Science, or Other. Format: CATEGORY: fact",
    output_key="categorized",
)

# ─── Stage 3: Summarize ──────────────────────────────────────────
summarizer = Agent(
    name="summarizer",
    model="gemini-2.5-flash",
    instruction="Write a 2-sentence executive summary from the categorized facts. Be professional.",
)

# ─── Compose into pipeline ───────────────────────────────────────
pipeline = SequentialAgent(
    name="research_pipeline",
    sub_agents=[extractor, categorizer, summarizer],  # Order matters!
)
print(f"Pipeline: {" → ".join(a.name for a in pipeline.sub_agents)}")


In [ ]:
async def pipeline_demo():
    svc = InMemorySessionService()
    s = await svc.create_session(app_name="t", user_id="u", session_id="seq")
    r = Runner(agent=pipeline, app_name="t", session_service=svc)
    m = Content(role="user", parts=[Part.from_text(
        text="Google DeepMind announced a 1000-qubit quantum processor. Tesla stock dropped 8% after missing delivery targets. NASA confirmed water ice on the Moons south pole."
    )])
    print("Sequential Pipeline Demo\n")
    for e in r.run(user_id="u", session_id=s.id, new_message=m):
        if e.content and e.content.parts and e.author != "user":
            for p in e.content.parts:
                if p.text: print(f"--- [{e.author}] ---\n{p.text[:400]}\n")
await pipeline_demo()


## 4.3 ParallelAgent — Concurrent Execution

Runs ALL sub-agents **at the same time** with the same input.
Best for getting multiple perspectives on one topic.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# PARALLEL ANALYSIS — Three perspectives on one topic
# ═══════════════════════════════════════════════════════════════════

tech = Agent(name="tech_analyst", model="gemini-2.5-flash",
    instruction="Analyze from a TECHNICAL perspective. 2-3 sentences.", output_key="tech_view")
biz = Agent(name="biz_analyst", model="gemini-2.5-flash",
    instruction="Analyze from a BUSINESS perspective. 2-3 sentences.", output_key="biz_view")
risk = Agent(name="risk_analyst", model="gemini-2.5-flash",
    instruction="Analyze from a RISK perspective. 2-3 sentences.", output_key="risk_view")

panel = ParallelAgent(name="analysis_panel", sub_agents=[tech, biz, risk])
print(f"Parallel panel: {[a.name for a in panel.sub_agents]}")

async def parallel_demo():
    svc = InMemorySessionService()
    s = await svc.create_session(app_name="t", user_id="u", session_id="par")
    r = Runner(agent=panel, app_name="t", session_service=svc)
    m = Content(role="user", parts=[Part.from_text(
        text="A startup is building a decentralized social network on blockchain with AI moderation."
    )])
    for e in r.run(user_id="u", session_id=s.id, new_message=m):
        if e.content and e.content.parts and e.author != "user":
            for p in e.content.parts:
                if p.text: print(f"[{e.author}]: {p.text[:300]}\n")
await parallel_demo()


## 4.4 Sub-Agent Routing — LLM Chooses the Specialist

This is the most flexible pattern: put specialist agents in a parent
agent's `sub_agents` list. The parent LLM decides which specialist
to transfer to based on the user's request.

**Internal mechanism:** ADK auto-creates a `transfer_to_agent` tool
for each sub-agent. The parent LLM "calls" this tool to delegate.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# LLM ROUTING — Parent agent delegates to specialists
# ═══════════════════════════════════════════════════════════════════

# Define specialist agents (each with focused instructions)
weather_sp = Agent(name="weather_specialist", model="gemini-2.5-flash",
    instruction="Answer weather questions. Be precise about temperature, conditions.",
    description="Handles weather queries for any city")

math_sp = Agent(name="math_specialist", model="gemini-2.5-flash",
    instruction="Solve math problems step-by-step. Show your work clearly.",
    description="Solves mathematical problems and equations")

code_sp = Agent(name="code_specialist", model="gemini-2.5-flash",
    instruction="Write and explain code with markdown code blocks. Include comments.",
    description="Writes and explains code in any language")

# The orchestrator — its LLM decides who to delegate to
router = Agent(name="orchestrator", model="gemini-2.5-flash",
    instruction="""Route user requests to the right specialist:
  - weather_specialist: weather, climate, temperature
  - math_specialist: calculations, equations, math
  - code_specialist: programming, debugging, code
Handle general questions yourself.""",
    sub_agents=[weather_sp, math_sp, code_sp],  # ← Specialists available to delegate to
)
print(f"Router with {len(router.sub_agents)} specialists: {[a.name for a in router.sub_agents]}")

async def route_demo(query):
    svc = InMemorySessionService()
    s = await svc.create_session(app_name="t", user_id="u", session_id="rt")
    r = Runner(agent=router, app_name="t", session_service=svc)
    m = Content(role="user", parts=[Part.from_text(text=query)])
    print(f"Q: {query}\n")
    for e in r.run(user_id="u", session_id=s.id, new_message=m):
        if e.content and e.content.parts and e.author != "user":
            for p in e.content.parts:
                if p.text: print(f"[{e.author}]: {p.text[:300]}")
                if p.function_call: print(f"  → Routing to: {p.function_call.name}")

await route_demo("Solve the quadratic equation: 3x^2 + 7x - 6 = 0")


## Multi-Agent Decision Guide

| You want... | Use | Example |
|-------------|-----|--------|
| Fixed A→B→C pipeline | `SequentialAgent` | Extract→Clean→Analyze |
| N independent tasks | `ParallelAgent` | Research 3 topics at once |
| LLM decides who handles | `sub_agents` on `Agent` | Route to specialist by topic |
| Complex workflow | Nest them! | Seq(Parallel(A,B), C) |

**Next:** [05_callbacks_memory_state.ipynb](./05_callbacks_memory_state.ipynb) — observability and persistence.
